In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import pickle
import time

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [8]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
print("✅ NLTK data downloaded!")

✅ NLTK data downloaded!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
# Load the dataset
df = pd.read_csv('data/spam-messages.csv', encoding='latin-1')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Dataset shape: (138815, 1)

Columns: ['label,text,URL,EMAIL,PHONE,lang,,,,,']


,"label,text,URL,EMAIL,PHONE,lang,,,,,"
0,"ham,Your opinion about me? 1. Over 2. Jada 3. ..."
1,"ham,What's up? Do you want me to come online? ..."
2,"ham,So u workin overtime nigpun?,No,No,No,en,,,,,"
3,"ham,""Also sir, i sent you an email about how t..."
4,"spam,Please Stay At Home. To encourage the not..."


In [10]:
# Parse the CSV properly - it seems to be comma-separated in a single column
# Let's split the first column by comma
if len(df.columns) == 1:
    # Split the column by comma
    df_split = df.iloc[:, 0].str.split(',', n=1, expand=True)
    df = pd.DataFrame({
        'label': df_split[0],
        'text': df_split[1]
    })
    print("✅ Data parsed successfully!")
else:
    # Use first two columns
    df = df.iloc[:, :2]
    df.columns = ['label', 'text']

# Clean the data
df = df.dropna()
df = df.drop_duplicates()

print(f"\nCleaned dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
df.head(10)

✅ Data parsed successfully!

Cleaned dataset shape: (138739, 2)

Label distribution:
label
ham       78321
spam      60393
Spam         23
ham"""        2
Name: count, dtype: int64


,label,text
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...
1,ham,What's up? Do you want me to come online? If y...
2,ham,"So u workin overtime nigpun?,No,No,No,en,,,,,"
3,ham,"""Also sir, i sent you an email about how to lo..."
4,spam,Please Stay At Home. To encourage the notion o...
5,spam,BankOfAmerica Alert 137943. Please follow http...
6,ham,Sorry dude. Dont know how i forgot. Even after...
7,ham,I don't quite know what to do. I still can't g...
8,ham,Ok lor. Anyway i thk we cant get tickets now c...
9,ham,"Wat r u doing now?,No,No,No,en,,,,,"


In [11]:
# Text Preprocessing Function
def preprocess_text(text):
    """
    Preprocess text for NLP:
    1. Convert to lowercase
    2. Remove URLs
    3. Remove emails
    4. Remove phone numbers
    5. Remove special characters and digits
    6. Remove extra whitespace
    7. Tokenize
    8. Remove stopwords
    9. Stemming
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove phone numbers
    text = re.sub(r'\d{10,}|\+\d+|\(\d+\)\s*\d+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    # Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    
    return ' '.join(tokens)

print("✅ Preprocessing function defined!")

# Test the function
sample_text = "URGENT! You have WON $1000! Call 1234567890 NOW! Visit http://example.com"
print(f"\nOriginal: {sample_text}")
print(f"Processed: {preprocess_text(sample_text)}")

✅ Preprocessing function defined!

Original: URGENT! You have WON $1000! Call 1234567890 NOW! Visit http://example.com
Processed: urgent call visit


In [12]:
# Apply preprocessing to all messages
print("Preprocessing messages... This may take a few minutes...")
start_time = time.time()

df['processed_text'] = df['text'].apply(preprocess_text)

# Remove empty processed texts
df = df[df['processed_text'].str.len() > 0]

end_time = time.time()
print(f"\n✅ Preprocessing completed in {end_time - start_time:.2f} seconds!")
print(f"Final dataset shape: {df.shape}")

# Show examples
print("\n📝 Sample preprocessed messages:")
for idx in range(min(3, len(df))):
    print(f"\nLabel: {df.iloc[idx]['label']}")
    print(f"Original: {df.iloc[idx]['text'][:100]}...")
    print(f"Processed: {df.iloc[idx]['processed_text'][:100]}...")

Preprocessing messages... This may take a few minutes...

✅ Preprocessing completed in 77.43 seconds!
Final dataset shape: (138316, 3)

📝 Sample preprocessed messages:

Label: ham
Original: Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl character 7. Not matu...
Processed: opinion jada kusruthi lovabl silent spl charact matur stylish simpl pl replynononoen...

Label: ham
Original: What's up? Do you want me to come online? If you are free we can talk sometime ,No,No,No,en,,,,,...
Processed: what want come onlin free talk sometim nononoen...

Label: ham
Original: So u workin overtime nigpun?,No,No,No,en,,,,,...
Processed: workin overtim nigpunnononoen...

✅ Preprocessing completed in 77.43 seconds!
Final dataset shape: (138316, 3)

📝 Sample preprocessed messages:

Label: ham
Original: Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl character 7. Not matu...
Processed: opinion jada kusruthi lovabl silent spl charact matur stylis

In [13]:
# Encode labels (ham=0, spam=1)
# Clean labels first - convert to lowercase and strip extra characters
df['label'] = df['label'].astype(str).str.lower().str.strip().str.replace('"', '')

# Filter to keep only valid ham/spam labels
df = df[df['label'].isin(['ham', 'spam'])]

# Now encode
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

# Check for any unmapped labels
print(f"Unique labels: {df['label'].unique()}")
print(f"\nLabel distribution:")
print(df['label_encoded'].value_counts())
print(f"\nHam: {(df['label_encoded']==0).sum()} | Spam: {(df['label_encoded']==1).sum()}")
print(f"\nFinal dataset shape after cleaning: {df.shape}")

Unique labels: ['ham' 'spam']

Label distribution:
label_encoded
0    77919
1    60397
Name: count, dtype: int64

Ham: 77919 | Spam: 60397

Final dataset shape after cleaning: (138316, 4)


In [14]:
# Split the data
X = df['processed_text']
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining set distribution:")
print(y_train.value_counts())
print(f"\nTest set distribution:")
print(y_test.value_counts())

Training set: 110652 samples
Test set: 27664 samples

Training set distribution:
label_encoded
0    62335
1    48317
Name: count, dtype: int64

Test set distribution:
label_encoded
0    15584
1    12080
Name: count, dtype: int64


In [15]:
# TF-IDF Vectorization - OPTIMIZED FOR MAXIMUM SPEED
print("⚡ Creating TF-IDF features with speed optimizations...")
print("="*80)
print("🎯 Strategy: Limited features + unigrams only for faster processing")
print("📊 Dataset: 138K+ messages | Target: Sub-minute training time")
print("="*80)

tfidf = TfidfVectorizer(
    max_features=10000,   # Limit to 10K features for speed
    ngram_range=(1, 1),   # Only unigrams (much faster than bigrams)
    min_df=3,             # Ignore rare terms (appear in < 3 docs)
    max_df=0.90,          # Ignore very common terms (> 90% docs)
    sublinear_tf=True,    # Use log scaling for TF
    strip_accents='unicode',
    lowercase=True,
    token_pattern=r'\b[a-z]{3,}\b'  # Only words with 3+ chars
)

vectorization_start = time.time()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
vectorization_time = time.time() - vectorization_start

print(f"\n✅ TF-IDF vectorization completed in {vectorization_time:.2f} seconds!")
print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Number of features: {len(tfidf.get_feature_names_out())}")
print(f"Matrix type: {type(X_train_tfidf).__name__} (memory efficient)")
print(f"Sparsity: {(1.0 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])):.2%}")
print(f"Memory usage: ~{X_train_tfidf.data.nbytes / (1024**2):.2f} MB")

⚡ Creating TF-IDF features with speed optimizations...
🎯 Strategy: Limited features + unigrams only for faster processing
📊 Dataset: 138K+ messages | Target: Sub-minute training time

✅ TF-IDF vectorization completed in 1.88 seconds!
Feature matrix shape: (110652, 10000)
Number of features: 10000
Matrix type: csr_matrix (memory efficient)
Sparsity: 99.93%
Memory usage: ~5.54 MB

✅ TF-IDF vectorization completed in 1.88 seconds!
Feature matrix shape: (110652, 10000)
Number of features: 10000
Matrix type: csr_matrix (memory efficient)
Sparsity: 99.93%
Memory usage: ~5.54 MB


In [16]:
# Train Multiple Models - ULTRA-FAST CONFIGURATION
print("\n🚀 SPEED-OPTIMIZED MODEL TRAINING")
print("="*80)
print("⚡ Focus: Maximum speed while maintaining accuracy")
print("🎯 Class Imbalance: Using class_weight='balanced' for all models")
print("💾 Memory: Using sparse matrices (CSR format)")
print("⏱️  Target: Complete training in under 2 minutes")
print("="*80)

models = {
    '⚡ SGDClassifier (Fastest)': SGDClassifier(
        loss='hinge',           # Linear SVM approximation
        penalty='l2',           # L2 regularization
        alpha=0.0001,          # Regularization strength
        max_iter=1000,         # Max epochs
        tol=1e-3,              # Stopping criterion
        random_state=42,
        class_weight='balanced',  # Handle imbalance
        n_jobs=-1,             # Use all CPU cores
        learning_rate='optimal', # Automatic learning rate
        early_stopping=False   # Train for full iterations
    ),
    '🚀 LinearSVC (Fast)': LinearSVC(
        penalty='l2',
        loss='squared_hinge',  # Faster than 'hinge'
        dual=False,            # Faster when n_samples > n_features
        tol=1e-3,
        C=1.0,
        max_iter=2000,
        random_state=42,
        class_weight='balanced'
    ),
    '📊 Naive Bayes (Baseline)': MultinomialNB(
        alpha=1.0
    ),
    '🔄 Logistic Regression': LogisticRegression(
        penalty='l2',
        solver='saga',         # Fast for large datasets
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
}

results = []
total_start_time = time.time()

print("\n🏃 Starting model training pipeline...")
print("="*80)

for name, model in models.items():
    print(f"\n{name}")
    print("-" * 60)
    
    # Training phase
    train_start = time.time()
    model.fit(X_train_tfidf, y_train)
    training_time = time.time() - train_start
    
    # Prediction phase
    pred_start = time.time()
    y_pred = model.predict(X_test_tfidf)
    pred_time = time.time() - pred_start
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Calculate speed metrics
    train_samples_per_sec = len(X_train) / training_time
    pred_samples_per_sec = len(X_test) / pred_time
    total_time = training_time + pred_time
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Train Time': training_time,
        'Pred Time': pred_time,
        'Total Time': total_time,
        'Train Speed': train_samples_per_sec,
        'Pred Speed': pred_samples_per_sec
    })
    
    # Print detailed results
    print(f"⏱️  Training Time: {training_time:.3f}s ({train_samples_per_sec:.0f} samples/sec)")
    print(f"⏱️  Prediction Time: {pred_time:.3f}s ({pred_samples_per_sec:.0f} samples/sec)")
    print(f"📊 Accuracy:  {accuracy*100:.2f}%")
    print(f"📊 Precision: {precision*100:.2f}%")
    print(f"📊 Recall:    {recall*100:.2f}%")
    print(f"📊 F1-Score:  {f1*100:.2f}%")
    print(f"✅ Total Time: {total_time:.3f}s")

total_training_time = time.time() - total_start_time

print("\n" + "="*80)
print(f"✅ ALL MODELS TRAINED SUCCESSFULLY!")
print(f"⏱️  Total Pipeline Time: {total_training_time:.2f} seconds")
print(f"📈 Models Trained: {len(models)}")
print(f"🎯 Training Samples: {len(X_train):,} | Test Samples: {len(X_test):,}")
print("="*80)


🚀 SPEED-OPTIMIZED MODEL TRAINING
⚡ Focus: Maximum speed while maintaining accuracy
🎯 Class Imbalance: Using class_weight='balanced' for all models
💾 Memory: Using sparse matrices (CSR format)
⏱️  Target: Complete training in under 2 minutes

🏃 Starting model training pipeline...

⚡ SGDClassifier (Fastest)
------------------------------------------------------------
⏱️  Training Time: 0.192s (577110 samples/sec)
⏱️  Prediction Time: 0.001s (20446031 samples/sec)
📊 Accuracy:  77.11%
📊 Precision: 73.15%
📊 Recall:    75.18%
📊 F1-Score:  74.15%
✅ Total Time: 0.193s

🚀 LinearSVC (Fast)
------------------------------------------------------------
⏱️  Training Time: 0.192s (577110 samples/sec)
⏱️  Prediction Time: 0.001s (20446031 samples/sec)
📊 Accuracy:  77.11%
📊 Precision: 73.15%
📊 Recall:    75.18%
📊 F1-Score:  74.15%
✅ Total Time: 0.193s

🚀 LinearSVC (Fast)
------------------------------------------------------------
⏱️  Training Time: 1.432s (77292 samples/sec)
⏱️  Prediction Time: 0.00

In [17]:
# Compare Results - Speed & Accuracy Analysis
results_df = pd.DataFrame(results)

print("\n" + "="*80)
print("📊 SPEED-OPTIMIZED MODEL COMPARISON (138K+ Messages)")
print("="*80)

# Create formatted table
display_df = results_df.copy()
display_df['Accuracy'] = display_df['Accuracy'].apply(lambda x: f"{x*100:.2f}%")
display_df['Precision'] = display_df['Precision'].apply(lambda x: f"{x*100:.2f}%")
display_df['Recall'] = display_df['Recall'].apply(lambda x: f"{x*100:.2f}%")
display_df['F1-Score'] = display_df['F1-Score'].apply(lambda x: f"{x*100:.2f}%")
display_df['Train Time'] = display_df['Train Time'].apply(lambda x: f"{x:.3f}s")
display_df['Total Time'] = display_df['Total Time'].apply(lambda x: f"{x:.3f}s")

print(display_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Train Time', 'Total Time']].to_string(index=False))

# Sort by F1-Score
results_sorted = results_df.sort_values('F1-Score', ascending=False)
best_model_name = results_sorted.iloc[0]['Model']

print("\n" + "="*80)
print("🏆 BEST MODEL (by F1-Score)")
print("="*80)
print(f"Model: {best_model_name}")
print(f"├─ Accuracy:  {results_sorted.iloc[0]['Accuracy']*100:.2f}%")
print(f"├─ Precision: {results_sorted.iloc[0]['Precision']*100:.2f}%")
print(f"├─ Recall:    {results_sorted.iloc[0]['Recall']*100:.2f}%")
print(f"├─ F1-Score:  {results_sorted.iloc[0]['F1-Score']*100:.2f}%")
print(f"└─ Training:  {results_sorted.iloc[0]['Train Time']:.3f}s ({results_sorted.iloc[0]['Train Speed']:.0f} samples/sec)")

# Fastest model
fastest_idx = results_df['Train Time'].idxmin()
fastest_model = results_df.loc[fastest_idx]

print("\n⚡ FASTEST MODEL (by Training Time)")
print("="*80)
print(f"Model: {fastest_model['Model']}")
print(f"├─ Training:  {fastest_model['Train Time']:.3f}s ({fastest_model['Train Speed']:.0f} samples/sec)")
print(f"├─ F1-Score:  {fastest_model['F1-Score']*100:.2f}%")
print(f"└─ Accuracy:  {fastest_model['Accuracy']*100:.2f}%")

# Speed comparison
if fastest_idx != results_sorted.index[0]:
    speedup = results_sorted.iloc[0]['Train Time'] / fastest_model['Train Time']
    f1_diff = (results_sorted.iloc[0]['F1-Score'] - fastest_model['F1-Score']) * 100
    print(f"\n💡 Trade-off: Best model is {speedup:.2f}x slower but {f1_diff:+.2f}% better F1-Score")

# SGDClassifier vs LinearSVC comparison
print("\n" + "="*80)
print("🔍 SGDClassifier vs LinearSVC Speed Comparison")
print("="*80)

sgd_row = results_df[results_df['Model'].str.contains('SGD', case=False)]
svc_row = results_df[results_df['Model'].str.contains('LinearSVC', case=False)]

if not sgd_row.empty and not svc_row.empty:
    sgd_time = sgd_row.iloc[0]['Train Time']
    svc_time = svc_row.iloc[0]['Train Time']
    sgd_f1 = sgd_row.iloc[0]['F1-Score']
    svc_f1 = svc_row.iloc[0]['F1-Score']
    
    print(f"SGDClassifier:  {sgd_time:.3f}s | F1={sgd_f1*100:.2f}%")
    print(f"LinearSVC:      {svc_time:.3f}s | F1={svc_f1*100:.2f}%")
    print(f"Speedup:        {svc_time/sgd_time:.2f}x faster" if sgd_time < svc_time else f"{sgd_time/svc_time:.2f}x faster")

# Save results
results_df.to_csv('cleaned_dataset/spam_model_comparison.csv', index=False)
print(f"\n✅ Results saved to 'cleaned_dataset/spam_model_comparison.csv'")
print("="*80)

# LinearSVC vs SGDClassifier comparison
print(f"\n🔍 LINEAR MODEL COMPARISON:")
linear_svc = results_df[results_df['Model'].str.contains('LinearSVC')]
sgd = results_df[results_df['Model'].str.contains('SGD')]

if not linear_svc.empty and not sgd.empty:
    print(f"\nLinearSVC:")
    print(f"   F1-Score: {linear_svc.iloc[0]['F1-Score']:.4f}")
    print(f"   Training: {linear_svc.iloc[0]['Training Time (s)']:.2f}s")
    print(f"\nSGDClassifier:")
    print(f"   F1-Score: {sgd.iloc[0]['F1-Score']:.4f}")
    print(f"   Training: {sgd.iloc[0]['Training Time (s)']:.2f}s")
    
    speed_diff = linear_svc.iloc[0]['Training Time (s)'] / sgd.iloc[0]['Training Time (s)']
    accuracy_diff = (sgd.iloc[0]['F1-Score'] - linear_svc.iloc[0]['F1-Score']) * 100
    
    print(f"\n   SGDClassifier is {speed_diff:.2f}x {'faster' if speed_diff > 1 else 'slower'} than LinearSVC")
    print(f"   F1-Score difference: {accuracy_diff:+.2f}%")

# Save results
results_df.to_csv('cleaned_dataset/spam_model_comparison.csv', index=False)
print("\n✅ Results saved to 'cleaned_dataset/spam_model_comparison.csv'")


📊 SPEED-OPTIMIZED MODEL COMPARISON (138K+ Messages)
                    Model Accuracy Precision Recall F1-Score Train Time Total Time
⚡ SGDClassifier (Fastest)   77.11%    73.15% 75.18%   74.15%     0.192s     0.193s
       🚀 LinearSVC (Fast)   79.20%    75.44% 77.64%   76.52%     1.432s     1.433s
 📊 Naive Bayes (Baseline)   76.86%    71.32% 78.62%   74.79%     0.017s     0.020s
    🔄 Logistic Regression   78.77%    74.79% 77.52%   76.13%     1.377s     1.380s

🏆 BEST MODEL (by F1-Score)
Model: 🚀 LinearSVC (Fast)
├─ Accuracy:  79.20%
├─ Precision: 75.44%
├─ Recall:    77.64%
├─ F1-Score:  76.52%
└─ Training:  1.432s (77292 samples/sec)

⚡ FASTEST MODEL (by Training Time)
Model: 📊 Naive Bayes (Baseline)
├─ Training:  0.017s (6651401 samples/sec)
├─ F1-Score:  74.79%
└─ Accuracy:  76.86%

💡 Trade-off: Best model is 86.06x slower but +1.73% better F1-Score

🔍 SGDClassifier vs LinearSVC Speed Comparison
SGDClassifier:  0.192s | F1=74.15%
LinearSVC:      1.432s | F1=76.52%
Speedup:      

KeyError: 'Training Time (s)'

In [ ]:
# Visualize Results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy
ax1 = axes[0, 0]
results_df.plot(x='Model', y='Accuracy', kind='barh', ax=ax1, color='skyblue', legend=False)
ax1.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_xlabel('Accuracy')
ax1.grid(axis='x', alpha=0.3)

# Precision, Recall, F1-Score
ax2 = axes[0, 1]
results_df.plot(x='Model', y=['Precision', 'Recall', 'F1-Score'], kind='barh', ax=ax2)
ax2.set_title('Precision, Recall & F1-Score Comparison', fontsize=14, fontweight='bold')
ax2.set_xlabel('Score')
ax2.legend(loc='lower right')
ax2.grid(axis='x', alpha=0.3)

# F1-Score
ax3 = axes[1, 0]
results_df.plot(x='Model', y='F1-Score', kind='barh', ax=ax3, color='coral', legend=False)
ax3.set_title('F1-Score Comparison', fontsize=14, fontweight='bold')
ax3.set_xlabel('F1-Score')
ax3.grid(axis='x', alpha=0.3)

# Training Time
ax4 = axes[1, 1]
results_df.plot(x='Model', y='Training Time (s)', kind='barh', ax=ax4, color='lightgreen', legend=False)
ax4.set_title('Training Time Comparison', fontsize=14, fontweight='bold')
ax4.set_xlabel('Time (seconds)')
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('cleaned_dataset/spam_model_comparison.png', dpi=300, bbox_inches='tight')
print("✅ Visualization saved to 'cleaned_dataset/spam_model_comparison.png'")
plt.show()

In [ ]:
# Detailed Analysis of Best Model
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_tfidf)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

print("="*80)
print(f"DETAILED ANALYSIS - {best_model_name.upper()}")
print("="*80)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Ham', 'Spam']))

# Visualize Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.savefig('cleaned_dataset/spam_confusion_matrix.png', dpi=300, bbox_inches='tight')
print("\n✅ Confusion matrix saved to 'cleaned_dataset/spam_confusion_matrix.png'")
plt.show()

# Additional Metrics
tn, fp, fn, tp = cm.ravel()
print("\nAdditional Metrics:")
print(f"True Negatives (Ham predicted as Ham): {tn}")
print(f"False Positives (Ham predicted as Spam): {fp}")
print(f"False Negatives (Spam predicted as Ham): {fn}")
print(f"True Positives (Spam predicted as Spam): {tp}")
print(f"\nFalse Positive Rate: {fp/(fp+tn):.4f}")
print(f"False Negative Rate: {fn/(fn+tp):.4f}")

In [19]:
# Save the best model and vectorizer
print("Saving model and vectorizer...")

# Get best model from results
results_sorted = results_df.sort_values('F1-Score', ascending=False)
best_model_name = results_sorted.iloc[0]['Model']
best_model = models[best_model_name]

# Save model
with open('cleaned_dataset/spam_detection_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f"✅ Model ({best_model_name}) saved to 'cleaned_dataset/spam_detection_model.pkl'")

# Save vectorizer
with open('cleaned_dataset/spam_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("✅ TF-IDF vectorizer saved to 'cleaned_dataset/spam_tfidf_vectorizer.pkl'")

# Save preprocessing info
model_info = {
    'model_name': best_model_name,
    'accuracy': results_sorted.iloc[0]['Accuracy'],
    'precision': results_sorted.iloc[0]['Precision'],
    'recall': results_sorted.iloc[0]['Recall'],
    'f1_score': results_sorted.iloc[0]['F1-Score'],
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'features': len(tfidf.get_feature_names_out())
}

with open('cleaned_dataset/spam_model_info.pkl', 'wb') as f:
    pickle.dump(model_info, f)
print("✅ Model info saved to 'cleaned_dataset/spam_model_info.pkl'")

print("\n" + "="*80)
print("✅ MODEL TRAINING AND SAVING COMPLETED SUCCESSFULLY!")
print("="*80)

Saving model and vectorizer...
✅ Model (🚀 LinearSVC (Fast)) saved to 'cleaned_dataset/spam_detection_model.pkl'
✅ TF-IDF vectorizer saved to 'cleaned_dataset/spam_tfidf_vectorizer.pkl'
✅ Model info saved to 'cleaned_dataset/spam_model_info.pkl'

✅ MODEL TRAINING AND SAVING COMPLETED SUCCESSFULLY!


In [ ]:
# Test the model with custom messages
def predict_message(message):
    """Predict if a message is spam or ham"""
    # Preprocess
    processed = preprocess_text(message)
    
    # Vectorize
    vectorized = tfidf.transform([processed])
    
    # Predict
    prediction = best_model.predict(vectorized)[0]
    probability = best_model.predict_proba(vectorized)[0]
    
    result = {
        'message': message,
        'prediction': 'SPAM' if prediction == 1 else 'HAM',
        'is_spam': bool(prediction == 1),
        'spam_probability': float(probability[1]),
        'ham_probability': float(probability[0]),
        'confidence': float(max(probability))
    }
    
    return result

# Test with sample messages
test_messages = [
    "Hey! How are you? Want to meet for coffee?",
    "URGENT! You have won $10000! Click here now to claim your prize!",
    "Can you pick up some milk on your way home?",
    "FREE!!! Call now to get exclusive offers. Limited time only!",
    "Meeting is scheduled for 3 PM tomorrow in conference room"
]

print("\n" + "="*80)
print("🧪 TESTING MODEL WITH SAMPLE MESSAGES")
print("="*80)

for msg in test_messages:
    result = predict_message(msg)
    print(f"\nMessage: {msg}")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']*100:.2f}%")
    print(f"Spam Probability: {result['spam_probability']*100:.2f}%")
    print("-" * 80)


🧪 TESTING MODEL WITH SAMPLE MESSAGES

Message: Hey! How are you? Want to meet for coffee?
Prediction: HAM
Confidence: 50.80%
Spam Probability: 49.20%
--------------------------------------------------------------------------------

Message: URGENT! You have won $10000! Click here now to claim your prize!
Prediction: SPAM
Confidence: 81.68%
Spam Probability: 81.68%
--------------------------------------------------------------------------------

Message: Can you pick up some milk on your way home?
Prediction: HAM
Confidence: 53.36%
Spam Probability: 46.64%
--------------------------------------------------------------------------------

Message: FREE!!! Call now to get exclusive offers. Limited time only!
Prediction: SPAM
Confidence: 54.59%
Spam Probability: 54.59%
--------------------------------------------------------------------------------

Message: Meeting is scheduled for 3 PM tomorrow in conference room
Prediction: HAM
Confidence: 59.77%
Spam Probability: 40.23%
---------------

: 